In [ ]:
import argparse
import os
import json
import numpy as np
from colabdesign import mk_afdesign_model, clear_mem
import time
import pickle

def load_settings(settings_path):
    with open(settings_path) as f:
        return json.load(f)


def calc_fixed_conservation(seq, masked_seq):
    canonical = 'ACDEFGHIKLMNPQRSTVWY'
    matches = sum(1 for s, m in zip(seq, masked_seq) if m in canonical and s == m)
    total = sum(1 for m in masked_seq if m in canonical)
    return matches / total if total > 0 else 0.0

def initialize_model(settings):
    clear_mem()
    af_model = mk_afdesign_model(protocol="binder", 
                                use_multimer=True,
                                data_dir=settings["data_dir"],
                                masked_seq=settings["masked_seq"],
                                initialise_seq=settings["initialise_seq"],
                                force_fixed=settings["force_fixed"])
    
    af_model.prep_inputs(pdb_filename=settings["input_pdb"],
                        chain=settings["chains"],
                        hotspot=settings["hotspot_residues"],
                        )
    # update contact distances
    af_model.opt["con"].update({"num":settings["intra_contact_number"],"cutoff":settings["intra_contact_distance"],"binary":False,"seqsep":9})
    af_model.opt["i_con"].update({"num":settings["inter_contact_number"],"cutoff":settings["inter_contact_distance"],"binary":False})

    af_model.opt["weights"].update({
        "pae": settings["weights"]["pae_intra"],
        "plddt": settings["weights"]["plddt"],
        "i_pae": settings["weights"]["pae_inter"],
        "con": settings["weights"]["con_intra"],
        "i_con": settings["weights"]["con_inter"],
        "rg": settings["weights"]["rg"],
        "i_ptm": settings["weights"]["iptm"],
        "ptmenergy": settings["weights"]["ptmenergy"],
    })
    return af_model

def run_optimization(af_model, settings, seed):    
    # Stage 1: Initial Logits
    print("Stage 1: Test Logits")
    af_model.design_logits(iters=5, e_soft=0.9, ramp_recycles=False, save_best=True)
    initial_plddt = round(np.mean(af_model._tmp["best"]["aux"]["plddt"][-af_model._binder_len]),2)
    
    if initial_plddt < 0.65:
        print("Initial trajectory pLDDT bad: "+str(initial_plddt))
        return None, None, None
    else:
        print("Initial trajectory pLDDT good, continuing: "+str(initial_plddt))
    # # Stage 2: Additional Logits
    # print("Stage 1: Additional Logits Optimisation")
    # af_model.design_logits(iters=settings["iterations"]["logit"], e_soft=1.0, ramp_recycles=False, save_best=True)
    
    # # Stage 3: Soft iterations
    # print('Starting soft iterations')
    # af_model.design_soft(iters=settings["iterations"]["soft"], e_temp=1e-2, ramp_recycles=False, save_best=True)
    
    # # Stage 4: Hard iterations
    # af_model.clear_best()
    # print('Starting hard iterations:')
    # af_model.design_hard(iters=settings["iterations"]["hard"], temp=1e-2, dropout=False, ramp_recycles=False, save_best=True)
    
    # # Stage 5: Semigreedy
    # print('Starting semigreedy')
    # af_model.design_pssm_semigreedy(soft_iters=0, hard_iters=settings["iterations"]["semigreedy"], tries=10, dropout=False, num_models=1, ramp_models=False, save_best=True)
    
    return af_model

def main():

    settings = load_settings("./IgA.json")
    os.makedirs(settings["output"]["colabdesign_output_directory"], exist_ok=True)

    before_time = time.time()
    print(f'Time before any designs: {before_time}')
    seed = np.random.randint(0, 999999)
    af_model = initialize_model(settings)
    run_optimization(af_model, settings, seed)
    return af_model
af_model = main()